### 자연어 문의 접수 서비스

사용자가 자연어로 문의를 남길 수 있는 서비스를 제공한다.

### 데이터 요구사항

데이터베이스와 샘플 데이터 생성에는 AI를 활용한다.

유저, 구매 내역, 문의의 세 테이블을 사용한다.
상품 정보는 구매 내역 테이블에 함께 저장하며, 별도의 상품 테이블로 분리하지 않아도 된다.

| 테이블 | 저장할 정보 |
|---|---|
| 유저 | 사용자 정보 |
| 구매 내역 | 구매한 사용자와 상품 등 구매 정보 |
| 문의 | 문의 내용과 유형, 문의한 사용자, 관련 구매 내역이 있으면 해당 구매와의 연결 정보 |

### 기능 요구사항

- 문의하는 사용자의 ID가 주어지며, 로그인 기능은 구현하지 않는다.
- 사용자가 자연어로 입력한 문의 내용을 파악하고 문의 유형을 분류한다.
- 구매 관련 문의이면 해당 사용자의 관련 구매 내역을 조회해 보여준다.
- 관련 구매가 여러 건이거나 찾을 수 없으면 사용자에게 추가 정보를 요청한다.
- 구매 관련 문의는 해당 구매 내역과 연결해 저장할 수 있어야 한다.
- 일반 문의는 구매 내역과 연결하지 않고 저장할 수 있어야 한다.

### 이용 예시

> 사용자: 지난번에 산 키보드 배송이 안 왔어요.
>
> 서비스: 키보드 구매 내역이 두 건 있습니다. 어떤 구매에 대한 문의인가요?
>
> 사용자: 지난주에 구매한 거요.

서비스는 해당 구매 내역을 보여주고, 그 구매 건에 배송 문의를 등록 후 사용자에게 완료 메시지를 남긴다.

In [1]:
import json
import os
from datetime import date
from typing import Literal
from urllib.parse import quote
from uuid import uuid4

import psycopg
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from psycopg.rows import dict_row